In [ ]:
# =============================================================================
# SYSTEM CLEANUP - RUN THIS FIRST
# =============================================================================

import os, gc, shutil, glob

def quick_cleanup():
    """Quick cleanup before execution."""
    print("CLEANUP...")
    gc.collect()
    
    try:
        import torch
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
            mem = torch.cuda.memory_allocated() / 1e9
            print(f"CUDA: {mem:.2f} GB allocated")
    except: pass
    
    for pattern in ['**/__pycache__', '**/.ipynb_checkpoints']:
        for p in glob.glob(pattern, recursive=True):
            try: shutil.rmtree(p)
            except: pass
    
    total, used, free = shutil.disk_usage('/')
    print(f"Disk space: {free/1e9:.1f} GB free")
    print("Ready!")

quick_cleanup()

In [ ]:
# =============================================================================
# IMPORTS & CONFIGURATION
# =============================================================================
import os
import numpy as np
import pandas as pd
import warnings
warnings.filterwarnings('ignore')

# Sklearn
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.metrics import accuracy_score, f1_score, classification_report, confusion_matrix
from sklearn.linear_model import LogisticRegression
import joblib

# Gradient Boosting
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier

# PyTorch
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, TensorDataset
from torch.optim import Adam, AdamW
from torch.optim.lr_scheduler import OneCycleLR, CosineAnnealingWarmRestarts

# Optuna for hyperparameter optimization
import optuna
from optuna.trial import TrialState

# Visualization
import matplotlib.pyplot as plt
import seaborn as sns

# Configuration
SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')
print(f'PyTorch version: {torch.__version__}')
print(f'Optuna version: {optuna.__version__}')

## 1. Data Loading

In [ ]:
# =============================================================================
# PATHS CONFIGURATION - Multi-Branch Architecture
# =============================================================================
DATA_DIR = './data'
EMB_DIR = os.path.join(DATA_DIR, 'embeddings')

# File paths for multi-branch architecture
paths = {
    # Separate embeddings for multi-branch (3072d each)
    'train_text_emb': os.path.join(EMB_DIR, 'X_train_text_multilayer.npy'),
    'train_desc_emb': os.path.join(EMB_DIR, 'X_train_desc_multilayer.npy'),
    'kaggle_text_emb': os.path.join(EMB_DIR, 'X_kaggle_text_multilayer.npy'),
    'kaggle_desc_emb': os.path.join(EMB_DIR, 'X_kaggle_desc_multilayer.npy'),
    # Combined embeddings (backward compatibility)
    'train_embeddings_ml': os.path.join(EMB_DIR, 'X_train_multilayer_embeddings.npy'),
    'kaggle_embeddings_ml': os.path.join(EMB_DIR, 'X_kaggle_multilayer_embeddings.npy'),
    # Structured features (CSV from feature_engineering.ipynb)
    'train_features': os.path.join(DATA_DIR, 'train_features.csv'),
    'kaggle_features': os.path.join(DATA_DIR, 'test_features.csv'),
    # Labels
    'y_train': os.path.join(DATA_DIR, 'y_train.npy'),
}

# Check files exist
print("Checking required files for multi-branch architecture:")
for name, path in paths.items():
    if os.path.exists(path):
        print(f"  [OK] {name}: {path}")
    else:
        print(f"  [MISSING] {name}: {path}")

In [ ]:
# =============================================================================
# LOAD DATA - Multi-Branch Architecture
# =============================================================================
def clean_array(arr):
    """Clean NaN/inf values and convert to float32."""
    arr = np.asarray(arr)
    arr = np.nan_to_num(arr, nan=0.0, posinf=1e6, neginf=-1e6)
    return arr.astype(np.float32)

# Load SEPARATE embeddings for multi-branch architecture
USE_MULTI_BRANCH = os.path.exists(paths['train_text_emb'])

if USE_MULTI_BRANCH:
    print("Loading SEPARATE embeddings for multi-branch architecture...")
    X_train_text = clean_array(np.load(paths['train_text_emb']))
    X_train_desc = clean_array(np.load(paths['train_desc_emb']))
    X_kaggle_text = clean_array(np.load(paths['kaggle_text_emb']))
    X_kaggle_desc = clean_array(np.load(paths['kaggle_desc_emb']))
    print(f"   Tweet embeddings: {X_train_text.shape}")
    print(f"   Desc embeddings: {X_train_desc.shape}")
else:
    # Fallback: load combined and split (if separate files not available)
    print("WARNING: Separate embeddings not found, loading combined and splitting...")
    X_train_combined = clean_array(np.load(paths['train_embeddings_ml']))
    X_kaggle_combined = clean_array(np.load(paths['kaggle_embeddings_ml']))
    # Split in half (first half = text, second half = desc)
    half = X_train_combined.shape[1] // 2
    X_train_text = X_train_combined[:, :half]
    X_train_desc = X_train_combined[:, half:]
    X_kaggle_text = X_kaggle_combined[:, :half]
    X_kaggle_desc = X_kaggle_combined[:, half:]
    print(f"   Split combined embeddings: {X_train_text.shape} + {X_train_desc.shape}")

# Load structured features from CSV (feature_engineering.ipynb output)
train_df = pd.read_csv(paths['train_features'])
kaggle_df = pd.read_csv(paths['kaggle_features'])

# Top numeric features (based on LightGBM feature importance)
# Ordered by decreasing importance from feature_engineering.ipynb
TOP_FEATURES = [
    # TOP 6 features (importance > 500)
    'user_description_length',   # importance=736
    'tweets_per_favourites',     # importance=699
    'user_favourites_count',     # importance=646
    'user_statuses_count',       # importance=639
    'user_listed_count',         # importance=607
    'listed_per_status',         # importance=553
    # Log transforms (high correlation)
    'log_user_listed',           # corr=0.606
    'log_user_statuses',         # corr=0.439
    'log_user_favourites',
    # Engagement features
    'total_engagement', 'log_total_engagement',
    'retweet_count', 'favorite_count', 'reply_count', 'quote_count',
    'log_retweet_count', 'log_favorite_count',
    # Binary user features (very discriminative)
    'user_has_url',              # Observer=16%, Influencer=56%
    'user_has_banner',           # Observer=73%, Influencer=92%
    'user_has_location',         # Observer=58%, Influencer=75%
    'user_has_long_desc',
    'user_default_profile', 'user_default_profile_image',
    # Source device (very discriminative)
    'is_iphone', 'is_android', 'is_web', 'is_tweetdeck', 'is_bot_source',
    # Text features
    'tweet_length', 'word_count', 'uppercase_ratio',
    'hashtag_count', 'is_hashtag_heavy',
    'mention_count', 'is_mention_heavy',
    'emoji_count', 'is_emoji_heavy',
    'exclamation_count', 'question_count', 'has_multiple_exclamations',
    'url_count', 'has_url',
    # Content detection
    'is_reply', 'is_in_reply', 'is_reply_to_someone',
    'has_rt_qt', 'is_quote_status', 'has_quoted_status',
    'has_call_to_action', 'has_self_promotion', 'has_media_reference',
    # Entities
    'entities_hashtags', 'entities_urls', 'entities_mentions', 'entities_symbols',
    # Other
    'first_person_count', 'is_long_tweet', 'is_short_tweet'
]

# Filter to features that exist
available_features = [f for f in TOP_FEATURES if f in train_df.columns]
print(f"Using {len(available_features)} structured features")

# Extract and clean features
X_train_feat = clean_array(train_df[available_features].fillna(0).values)
X_kaggle_feat = clean_array(kaggle_df[available_features].fillna(0).values)

# Normalize features
from sklearn.preprocessing import StandardScaler
feat_scaler = StandardScaler()
X_train_feat = feat_scaler.fit_transform(X_train_feat)
X_kaggle_feat = feat_scaler.transform(X_kaggle_feat)

# Load labels
y_full = np.load(paths['y_train'])
le = LabelEncoder()
y_full = le.fit_transform(y_full)

# Store dimensions for model
TWEET_DIM = X_train_text.shape[1]
DESC_DIM = X_train_desc.shape[1]
FEAT_DIM = X_train_feat.shape[1]

print(f"\nData Shapes (Multi-Branch):")
print(f"   Tweet embeddings (train): {X_train_text.shape} -> TWEET_DIM={TWEET_DIM}")
print(f"   Desc embeddings (train): {X_train_desc.shape} -> DESC_DIM={DESC_DIM}")
print(f"   Structured features (train): {X_train_feat.shape} -> FEAT_DIM={FEAT_DIM}")
print(f"   Labels: {y_full.shape}, Classes: {np.unique(y_full)}")
print(f"   Class distribution: {np.bincount(y_full)}")

## 2. Advanced Feature Engineering

Inspired by Lab6 - Creation of combined features and interactions.

In [ ]:
# =============================================================================
# MULTI-BRANCH DATA PREPARATION
# =============================================================================

# For multi-branch architecture, we keep the 3 inputs SEPARATE:
# 1. Tweet embeddings (TWEET_DIM)
# 2. Description embeddings (DESC_DIM)
# 3. Structured features (FEAT_DIM)

print(f"Multi-Branch Data (3 separate inputs):")
print(f"   Tweet embeddings: {X_train_text.shape}")
print(f"   Desc embeddings: {X_train_desc.shape}")
print(f"   Structured features: {X_train_feat.shape}")

# Also create combined version for boosting models (XGBoost/LightGBM)
X_train_combined = np.hstack([X_train_text, X_train_desc, X_train_feat])
X_kaggle_combined = np.hstack([X_kaggle_text, X_kaggle_desc, X_kaggle_feat])

print(f"\nCombined (for boosting models):")
print(f"   Train: {X_train_combined.shape}")
print(f"   Kaggle: {X_kaggle_combined.shape}")

print(f"\nDimensions for Multi-Branch NN:")
print(f"   Tweet branch: {TWEET_DIM}")
print(f"   Desc branch: {DESC_DIM}")
print(f"   Features branch: {FEAT_DIM}")
print(f"   Total (combined): {X_train_combined.shape[1]}")

In [ ]:
# =============================================================================
# TRAIN/VAL SPLIT - Multi-Branch
# =============================================================================
from sklearn.model_selection import train_test_split

# Split all 3 inputs consistently using the same random state
# Combined (for boosting models)
X_train_comb, X_val_comb, y_train, y_val = train_test_split(
    X_train_combined, y_full, 
    test_size=0.15, 
    random_state=SEED, 
    stratify=y_full
)

# Tweet embeddings
X_train_text_split, X_val_text_split = train_test_split(
    X_train_text, test_size=0.15, random_state=SEED, stratify=y_full
)

# Description embeddings
X_train_desc_split, X_val_desc_split = train_test_split(
    X_train_desc, test_size=0.15, random_state=SEED, stratify=y_full
)

# Structured features
X_train_feat_split, X_val_feat_split = train_test_split(
    X_train_feat, test_size=0.15, random_state=SEED, stratify=y_full
)

print(f"Train/Val split (Multi-Branch):")
print(f"   Tweet: Train {X_train_text_split.shape}, Val {X_val_text_split.shape}")
print(f"   Desc: Train {X_train_desc_split.shape}, Val {X_val_desc_split.shape}")
print(f"   Feat: Train {X_train_feat_split.shape}, Val {X_val_feat_split.shape}")
print(f"   Labels: Train {np.bincount(y_train)}, Val {np.bincount(y_val)}")

## Configuration: Training or Loading Models

Configure here which models you want to train or load from saved checkpoints.

In [ ]:
# =============================================================================
# CONFIGURATION: CHOOSE WHICH MODELS TO TRAIN
# =============================================================================

# Configure here what you want to do
TRAIN_NEURAL_NETWORK = True   # Set to True to retrain the NN
TRAIN_XGBOOST = False          # Set to False to load from file
TRAIN_LIGHTGBM = False         # Set to False to load from file

# =============================================================================
# NEURAL NETWORK ARCHITECTURE
# =============================================================================
# Available options:
#   - "multibranch"  : 3 separate branches (tweet, desc, features) then fusion
#   - "simple"       : Fully connected with all features concatenated

MODEL_ARCHITECTURE = "simple"  # <- CHANGE HERE TO SWITCH

# Saved model paths
MODEL_DIR = 'models'
XGBOOST_PATH = os.path.join(MODEL_DIR, 'xgb_model.joblib')
LIGHTGBM_PATH = os.path.join(MODEL_DIR, 'lgbm_model.joblib')
SCALER_PATH = os.path.join(MODEL_DIR, 'scaler.joblib')

# NN path depends on architecture
NN_PATHS = {
    "multibranch": os.path.join(MODEL_DIR, 'influencer_model_multibranch.pt'),
    "simple": os.path.join(MODEL_DIR, 'influencer_model_simple.pt'),
}
NN_PATH = NN_PATHS[MODEL_ARCHITECTURE]

print("Training Configuration:")
print(f"   Architecture: {MODEL_ARCHITECTURE.upper()}")
print(f"   Neural Network: {'TRAIN' if TRAIN_NEURAL_NETWORK else 'LOAD'}")
print(f"   XGBoost: {'TRAIN' if TRAIN_XGBOOST else 'LOAD'}")
print(f"   LightGBM: {'TRAIN' if TRAIN_LIGHTGBM else 'LOAD'}")

# Check which models exist
print("\nChecking saved models...")
print(f"   XGBoost: {'Found' if os.path.exists(XGBOOST_PATH) else 'Not found'}")
print(f"   LightGBM: {'Found' if os.path.exists(LIGHTGBM_PATH) else 'Not found'}")
print(f"   Scaler: {'Found' if os.path.exists(SCALER_PATH) else 'Not found'}")
print(f"   Neural Network: {'Found' if os.path.exists(NN_PATH) else 'Not found'}")

# Warnings if needed
if not TRAIN_XGBOOST and not os.path.exists(XGBOOST_PATH):
    print("\nWARNING: XGBoost set to LOAD but file not found! Will train instead.")
    TRAIN_XGBOOST = True

if not TRAIN_LIGHTGBM and not os.path.exists(LIGHTGBM_PATH):
    print("\nWARNING: LightGBM set to LOAD but file not found! Will train instead.")
    TRAIN_LIGHTGBM = True

if (not TRAIN_XGBOOST or not TRAIN_LIGHTGBM) and not os.path.exists(SCALER_PATH):
    print("\nWARNING: Scaler not found but needed for boosting models!")
    print("   Will create new scaler - make sure to retrain boosting models!")
    TRAIN_XGBOOST = True
    TRAIN_LIGHTGBM = True

## Loading System Configured

The selective loading/training system is now in place.

The system will automatically:
- Load XGBoost/LightGBM from `models/` if flags are `False`
- Train them if flags are `True`
- Handle the scaler automatically

**Ready to use!** Configure the flags in the configuration cell and run the notebook normally.

## Usage Guide: Retraining Only the Neural Network

### Scenario 1: Train Everything from Scratch
```python
TRAIN_NEURAL_NETWORK = True
TRAIN_XGBOOST = True  
TRAIN_LIGHTGBM = True
```
Run all cells normally.

### Scenario 2: Retrain Only the Neural Network
```python
TRAIN_NEURAL_NETWORK = True   # Retrain the NN
TRAIN_XGBOOST = False          # Load from file
TRAIN_LIGHTGBM = False         # Load from file
```

**Workflow:**
1. Run Configuration cell - defines what to load/train
2. Run the boosting cell (modified) - loads XGBoost and LightGBM from files
3. Run NN configuration cell
4. Run training cell - **RETRAINS** the Neural Network
5. Run ensemble cell - generates ensemble with new weights
6. Run submission cell - generates the submission

### Scenario 3: Load Everything (No Training)
```python
TRAIN_NEURAL_NETWORK = False
TRAIN_XGBOOST = False
TRAIN_LIGHTGBM = False
```
Useful for:
- Quickly generating predictions
- Testing ensemble code
- Creating new submissions without retraining

**Note:** You must first run the notebook completely once to save the models in `models/`!

## 3. Neural Network Architecture (Lab4 + Lab5 Inspired)

Multi-branch architecture with:
- Dropout regularization (Lab4)
- Batch Normalization
- Residual connections (Lab5 Transformer concept)
- Multi-Sample Dropout (recommended in TODO.md)

In [ ]:
# =============================================================================
# NEURAL NETWORK ARCHITECTURES
# =============================================================================
# Two architectures available, configurable via MODEL_ARCHITECTURE

# -----------------------------------------------------------------------------
# Architecture 1: SIMPLE (Fully Connected)
# -----------------------------------------------------------------------------
class InfluencerModelSimple(nn.Module):
    """
    Simple Fully Connected Classifier.
    
    Architecture: input -> 512 -> 256 -> 128 -> 64 -> n_classes
    All features are concatenated as input.
    """
    
    def __init__(self, input_dim, n_classes=2, hidden_dims=[512, 256, 128, 64], dropout=0.3):
        super().__init__()
        
        layers = []
        prev_dim = input_dim
        
        for i, hidden_dim in enumerate(hidden_dims):
            layers.append(nn.Linear(prev_dim, hidden_dim))
            layers.append(nn.BatchNorm1d(hidden_dim))
            layers.append(nn.ReLU(inplace=True))
            # Decreasing dropout towards output
            drop_rate = dropout * (1 - i / (len(hidden_dims) + 1))
            layers.append(nn.Dropout(drop_rate))
            prev_dim = hidden_dim
        
        # Output layer
        layers.append(nn.Linear(prev_dim, n_classes))
        
        self.network = nn.Sequential(*layers)
        self._init_weights()
    
    def _init_weights(self):
        """He initialization for ReLU networks."""
        for m in self.modules():
            if isinstance(m, nn.Linear):
                nn.init.kaiming_normal_(m.weight, mode='fan_out', nonlinearity='relu')
                if m.bias is not None:
                    nn.init.zeros_(m.bias)
            elif isinstance(m, nn.BatchNorm1d):
                nn.init.ones_(m.weight)
                nn.init.zeros_(m.bias)
    
    def forward(self, x):
        """Forward pass - single concatenated input."""
        return self.network(x)


# -----------------------------------------------------------------------------
# Architecture 2: MULTI-BRANCH
# -----------------------------------------------------------------------------
class InfluencerModelMultiBranch(nn.Module):
    """
    Multi-Branch Classifier with separate processing for each input type.
    
    Architecture:
    - Tweet branch: tweet_dim -> hidden_tweet -> hidden_tweet//2
    - Desc branch: desc_dim -> hidden_desc -> hidden_desc//2  
    - Features branch: meta_dim -> hidden_feat -> hidden_feat//2
    - Fusion: combined -> fusion_dim -> 64 -> n_classes
    """
    
    def __init__(self, tweet_dim=3072, desc_dim=3072, meta_dim=40, 
                 hidden_tweet=512, hidden_desc=512, hidden_feat=128,
                 fusion_dim=256, n_classes=2, dropout=0.3):
        super().__init__()
        
        # Store dimensions for later use
        self.hidden_tweet_out = hidden_tweet // 2
        self.hidden_desc_out = hidden_desc // 2
        self.hidden_feat_out = hidden_feat // 2
        
        # Tweet branch (text embeddings)
        self.tweet_branch = nn.Sequential(
            nn.Linear(tweet_dim, hidden_tweet),
            nn.BatchNorm1d(hidden_tweet),
            nn.ReLU(inplace=True),
            nn.Dropout(dropout),
            nn.Linear(hidden_tweet, self.hidden_tweet_out),
            nn.BatchNorm1d(self.hidden_tweet_out),
            nn.ReLU(inplace=True),
        )
        
        # Description branch (user bio embeddings)
        self.desc_branch = nn.Sequential(
            nn.Linear(desc_dim, hidden_desc),
            nn.BatchNorm1d(hidden_desc),
            nn.ReLU(inplace=True),
            nn.Dropout(dropout),
            nn.Linear(hidden_desc, self.hidden_desc_out),
            nn.BatchNorm1d(self.hidden_desc_out),
            nn.ReLU(inplace=True),
        )
        
        # Features branch (structured features)
        self.meta_branch = nn.Sequential(
            nn.Linear(meta_dim, hidden_feat),
            nn.BatchNorm1d(hidden_feat),
            nn.ReLU(inplace=True),
            nn.Dropout(dropout * 0.5),
            nn.Linear(hidden_feat, self.hidden_feat_out),
            nn.BatchNorm1d(self.hidden_feat_out),
            nn.ReLU(inplace=True),
        )
        
        # Fusion head
        combined_dim = self.hidden_tweet_out + self.hidden_desc_out + self.hidden_feat_out
        self.head = nn.Sequential(
            nn.Linear(combined_dim, fusion_dim),
            nn.BatchNorm1d(fusion_dim),
            nn.ReLU(inplace=True),
            nn.Dropout(dropout),
            nn.Linear(fusion_dim, 64),
            nn.ReLU(inplace=True),
            nn.Linear(64, n_classes)
        )
        
        self._init_weights()
    
    def _init_weights(self):
        """He initialization for ReLU networks."""
        for m in self.modules():
            if isinstance(m, nn.Linear):
                nn.init.kaiming_normal_(m.weight, mode='fan_out', nonlinearity='relu')
                if m.bias is not None:
                    nn.init.zeros_(m.bias)
            elif isinstance(m, nn.BatchNorm1d):
                nn.init.ones_(m.weight)
                nn.init.zeros_(m.bias)
    
    def forward(self, tweet_emb, desc_emb, meta):
        """Forward pass - 3 separate inputs."""
        t = self.tweet_branch(tweet_emb)
        d = self.desc_branch(desc_emb)
        m = self.meta_branch(meta)
        x = torch.cat([t, d, m], dim=1)
        return self.head(x)


# -----------------------------------------------------------------------------
# FACTORY FUNCTION - Creates the right model based on MODEL_ARCHITECTURE
# -----------------------------------------------------------------------------
def create_model(architecture, **kwargs):
    """
    Factory function to create model based on chosen architecture.
    
    Args:
        architecture: "simple" or "multibranch"
        **kwargs: model hyperparameters
    
    Returns:
        nn.Module: The created model
    """
    if architecture == "simple":
        # For simple, we need input_dim = tweet + desc + feat
        input_dim = kwargs.get('tweet_dim', TWEET_DIM) + \
                    kwargs.get('desc_dim', DESC_DIM) + \
                    kwargs.get('meta_dim', FEAT_DIM)
        return InfluencerModelSimple(
            input_dim=input_dim,
            n_classes=kwargs.get('n_classes', 2),
            hidden_dims=kwargs.get('hidden_dims', [512, 256, 128, 64]),
            dropout=kwargs.get('dropout', 0.3)
        )
    
    elif architecture == "multibranch":
        return InfluencerModelMultiBranch(
            tweet_dim=kwargs.get('tweet_dim', TWEET_DIM),
            desc_dim=kwargs.get('desc_dim', DESC_DIM),
            meta_dim=kwargs.get('meta_dim', FEAT_DIM),
            hidden_tweet=kwargs.get('hidden_tweet', 512),
            hidden_desc=kwargs.get('hidden_desc', 512),
            hidden_feat=kwargs.get('hidden_feat', 128),
            fusion_dim=kwargs.get('fusion_dim', 256),
            n_classes=kwargs.get('n_classes', 2),
            dropout=kwargs.get('dropout', 0.3)
        )
    
    else:
        raise ValueError(f"Unknown architecture: {architecture}. Use 'simple' or 'multibranch'")


# -----------------------------------------------------------------------------
# Display selected architecture
# -----------------------------------------------------------------------------
print(f"Selected architecture: {MODEL_ARCHITECTURE.upper()}")
print("=" * 60)

# Create a test model to display architecture
_test_model = create_model(MODEL_ARCHITECTURE, 
                           tweet_dim=TWEET_DIM, 
                           desc_dim=DESC_DIM, 
                           meta_dim=FEAT_DIM)
print(_test_model)
print(f"\nTotal parameters: {sum(p.numel() for p in _test_model.parameters()):,}")
del _test_model

## 4. Training Loop (Lab4 + Lab8 Inspired)

With:
- AdamW optimizer with weight decay (Lab8)
- Learning rate scheduling (OneCycleLR)
- Early stopping
- Gradient clipping

In [ ]:
# =============================================================================
# DATASETS & DATALOADERS (Architecture-aware)
# =============================================================================

# -----------------------------------------------------------------------------
# Dataset for SIMPLE architecture (concatenated features)
# -----------------------------------------------------------------------------
class SimpleDataset(Dataset):
    """Dataset for simple FC model - all features concatenated."""
    def __init__(self, features, labels=None):
        self.features = torch.from_numpy(features).float()
        self.labels = None if labels is None else torch.from_numpy(labels).long()
    
    def __len__(self):
        return self.features.shape[0]
    
    def __getitem__(self, idx):
        if self.labels is None:
            return self.features[idx]
        return self.features[idx], self.labels[idx]


# -----------------------------------------------------------------------------
# Dataset for MULTI-BRANCH architecture (3 separate inputs)
# -----------------------------------------------------------------------------
class MultiBranchDataset(Dataset):
    """Dataset for multi-branch model with 3 separate inputs."""
    def __init__(self, tweet_emb, desc_emb, features, labels=None):
        self.tweet = torch.from_numpy(tweet_emb).float()
        self.desc = torch.from_numpy(desc_emb).float()
        self.features = torch.from_numpy(features).float()
        self.labels = None if labels is None else torch.from_numpy(labels).long()
    
    def __len__(self):
        return self.tweet.shape[0]
    
    def __getitem__(self, idx):
        if self.labels is None:
            return self.tweet[idx], self.desc[idx], self.features[idx]
        return self.tweet[idx], self.desc[idx], self.features[idx], self.labels[idx]


# -----------------------------------------------------------------------------
# FACTORY - Create datasets and loaders based on architecture
# -----------------------------------------------------------------------------
def create_datasets_and_loaders(architecture, batch_size=512):
    """
    Create appropriate datasets and dataloaders based on architecture.
    
    Returns:
        train_loader, val_loader, train_dataset, val_dataset
    """
    if architecture == "simple":
        # Concatenate all features for simple architecture
        X_train_all = np.hstack([X_train_text_split, X_train_desc_split, X_train_feat_split])
        X_val_all = np.hstack([X_val_text_split, X_val_desc_split, X_val_feat_split])
        
        train_dataset = SimpleDataset(X_train_all, y_train)
        val_dataset = SimpleDataset(X_val_all, y_val)
        
    elif architecture == "multibranch":
        train_dataset = MultiBranchDataset(
            X_train_text_split, X_train_desc_split, X_train_feat_split, y_train
        )
        val_dataset = MultiBranchDataset(
            X_val_text_split, X_val_desc_split, X_val_feat_split, y_val
        )
    
    else:
        raise ValueError(f"Unknown architecture: {architecture}")
    
    train_loader = DataLoader(
        train_dataset, batch_size=batch_size, shuffle=True,
        num_workers=4, pin_memory=True, persistent_workers=True
    )
    val_loader = DataLoader(
        val_dataset, batch_size=batch_size, shuffle=False,
        num_workers=4, pin_memory=True, persistent_workers=True
    )
    
    return train_loader, val_loader, train_dataset, val_dataset


# -----------------------------------------------------------------------------
# Create loaders based on MODEL_ARCHITECTURE
# -----------------------------------------------------------------------------
batch_size = 512  # Large batch for GPU

train_loader, val_loader, train_dataset, val_dataset = create_datasets_and_loaders(
    MODEL_ARCHITECTURE, batch_size
)

print(f"DataLoaders created for {MODEL_ARCHITECTURE.upper()} architecture:")
print(f"   Batch size: {batch_size}")
print(f"   Train batches: {len(train_loader)}")
print(f"   Val batches: {len(val_loader)}")

In [ ]:
# =============================================================================
# TRAINING UTILITIES (Architecture-aware)
# =============================================================================

def _forward_pass(model, batch, architecture, device):
    """
    Execute forward pass according to architecture.
    
    Returns:
        logits, labels (both on device)
    """
    if architecture == "simple":
        features, labels = batch
        features = features.to(device)
        labels = labels.to(device)
        logits = model(features)
    else:  # multibranch
        tweet, desc, features, labels = batch
        tweet = tweet.to(device)
        desc = desc.to(device)
        features = features.to(device)
        labels = labels.to(device)
        logits = model(tweet, desc, features)
    
    return logits, labels


def train_epoch(model, loader, optimizer, criterion, scheduler, device, 
                architecture, max_grad_norm=1.0):
    """Train for one epoch with gradient clipping - Architecture-aware."""
    model.train()
    total_loss = 0
    all_preds, all_labels = [], []
    
    for batch in loader:
        logits, labels = _forward_pass(model, batch, architecture, device)
        
        optimizer.zero_grad()
        loss = criterion(logits, labels)
        
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_grad_norm)
        
        optimizer.step()
        scheduler.step()
        
        total_loss += loss.item()
        all_preds.extend(torch.argmax(logits, dim=1).cpu().numpy())
        all_labels.extend(labels.cpu().numpy())
    
    acc = accuracy_score(all_labels, all_preds)
    f1 = f1_score(all_labels, all_preds, average='macro')
    return total_loss / len(loader), acc, f1


def evaluate(model, loader, criterion, device, architecture):
    """Evaluate model on validation set - Architecture-aware."""
    model.eval()
    total_loss = 0
    all_preds, all_labels, all_probs = [], [], []
    
    with torch.no_grad():
        for batch in loader:
            logits, labels = _forward_pass(model, batch, architecture, device)
            loss = criterion(logits, labels)
            
            total_loss += loss.item()
            probs = F.softmax(logits, dim=1)
            all_preds.extend(torch.argmax(logits, dim=1).cpu().numpy())
            all_labels.extend(labels.cpu().numpy())
            all_probs.extend(probs.cpu().numpy())
    
    acc = accuracy_score(all_labels, all_preds)
    f1 = f1_score(all_labels, all_preds, average='macro')
    return total_loss / len(loader), acc, f1, np.array(all_probs)


print(f"Training utilities ready for {MODEL_ARCHITECTURE.upper()} architecture")


def plot_training_history(history):
    """Plot training curves."""
    fig, axes = plt.subplots(1, 3, figsize=(15, 4))
    
    # Loss
    axes[0].plot(history['train_loss'], label='Train')
    axes[0].plot(history['val_loss'], label='Val')
    axes[0].set_xlabel('Epoch')
    axes[0].set_ylabel('Loss')
    axes[0].set_title('Loss Curve')
    axes[0].legend()
    axes[0].grid(True, alpha=0.3)
    
    # Accuracy
    axes[1].plot(history['train_acc'], label='Train')
    axes[1].plot(history['val_acc'], label='Val')
    axes[1].set_xlabel('Epoch')
    axes[1].set_ylabel('Accuracy')
    axes[1].set_title('Accuracy Curve')
    axes[1].legend()
    axes[1].grid(True, alpha=0.3)
    
    # F1 Score
    axes[2].plot(history['train_f1'], label='Train')
    axes[2].plot(history['val_f1'], label='Val')
    axes[2].set_xlabel('Epoch')
    axes[2].set_ylabel('F1 Score')
    axes[2].set_title('F1 Score Curve')
    axes[2].legend()
    axes[2].grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.show()

In [ ]:
# =============================================================================
# TRAINING CONFIGURATION (Architecture-aware)
# =============================================================================

# Hyperparameters
EPOCHS = 30
LR = 1e-3
WEIGHT_DECAY = 4.0
PATIENCE = 30

# Initialize model using factory function
model = create_model(
    MODEL_ARCHITECTURE,
    tweet_dim=TWEET_DIM,
    desc_dim=DESC_DIM,
    meta_dim=FEAT_DIM,
    n_classes=2,
    dropout=0.3
).to(device)

# Loss function with class weights (handle imbalance)
class_counts = np.bincount(y_train)
class_weights = torch.tensor([1.0 / c for c in class_counts], dtype=torch.float32).to(device)
class_weights = class_weights / class_weights.sum() * 2
criterion = nn.CrossEntropyLoss(weight=class_weights)

# AdamW optimizer
optimizer = AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)

# Learning rate scheduler
total_steps = len(train_loader) * EPOCHS
scheduler = OneCycleLR(
    optimizer, 
    max_lr=LR,
    total_steps=total_steps,
    pct_start=0.1,
    anneal_strategy='cos'
)

print(f"Training Configuration ({MODEL_ARCHITECTURE.upper()}):")
print(f"   Epochs: {EPOCHS}")
print(f"   Learning rate: {LR}")
print(f"   Weight decay: {WEIGHT_DECAY}")
print(f"   Class weights: {class_weights.cpu().numpy()}")
if MODEL_ARCHITECTURE == "multibranch":
    print(f"   Model branches: Tweet({TWEET_DIM}) + Desc({DESC_DIM}) + Feat({FEAT_DIM})")
else:
    print(f"   Input dimension: {TWEET_DIM + DESC_DIM + FEAT_DIM}")

## 4.1 Optuna Hyperparameter Optimization

Automatic hyperparameter optimization with Optuna:
- Learning rate, weight decay, dropout
- Branch dimensions (hidden_tweet, hidden_desc, hidden_feat)
- Fusion dimension
- Batch size

**Note:** Set `RUN_OPTUNA = True` to launch optimization (may take several hours).

In [ ]:
# =============================================================================
# OPTUNA CONFIGURATION
# =============================================================================

# Optuna only works with multibranch architecture
# For simple architecture, set hyperparameters manually in DEFAULT_PARAMS
if MODEL_ARCHITECTURE == "simple":
    RUN_OPTUNA = False
    print("Optuna disabled for SIMPLE architecture")
    print("   -> Modify DEFAULT_PARAMS for hyperparameters")
else:
    RUN_OPTUNA = True   # Set to True to run hyperparameter optimization

N_TRIALS = 5        # Number of trials (more = better but slower)
OPTUNA_EPOCHS = 10  # Fewer epochs per trial for speed
TIMEOUT = 3600 * 4  # 4 hours max

# Study storage (optional - save progress)
STUDY_NAME = "influencer_multibranch_v1"
STUDY_PATH = f"models/optuna_{STUDY_NAME}.db"

print(f"\nOptuna Configuration:")
print(f"   Architecture: {MODEL_ARCHITECTURE.upper()}")
print(f"   Run optimization: {RUN_OPTUNA}")
if RUN_OPTUNA:
    print(f"   Number of trials: {N_TRIALS}")
    print(f"   Epochs per trial: {OPTUNA_EPOCHS}")
    print(f"   Timeout: {TIMEOUT/3600:.1f} hours")

In [ ]:
# =============================================================================
# OPTUNA OBJECTIVE FUNCTION (ANTI-OVERFITTING VERSION)
# =============================================================================

# ULTRA-STRICT Anti-overfitting configuration
MAX_TRAIN_VAL_GAP = 0.01   # HARD REJECT if gap > 1%
GAP_TARGET = 0.005         # Target gap around 0.5%
GAP_PENALTY_WEIGHT = 10.0  # Aggressive penalty


def evaluate_f1(model, loader, device):
    """Compute F1 score on a data loader without gradients."""
    model.eval()
    all_preds, all_labels = [], []
    with torch.no_grad():
        for batch in loader:
            tweet, desc, features, labels = [b.to(device) for b in batch]
            logits = model(tweet, desc, features)
            all_preds.extend(torch.argmax(logits, dim=1).cpu().numpy())
            all_labels.extend(labels.cpu().numpy())
    return f1_score(all_labels, all_preds, average='macro')


def create_model_from_trial(trial):
    """Create a model with hyperparameters suggested by Optuna.
    
    ANTI-OVERFITTING: Constrained search space with smaller architectures.
    """
    
    # Architecture hyperparameters - CONSTRAINED to prevent overfitting
    hidden_tweet = trial.suggest_categorical('hidden_tweet', [256, 384, 512])
    hidden_desc = trial.suggest_categorical('hidden_desc', [256, 384, 512])
    hidden_feat = trial.suggest_categorical('hidden_feat', [64, 128])
    fusion_dim = trial.suggest_categorical('fusion_dim', [128, 256])
    
    # Higher minimum dropout for regularization
    dropout = trial.suggest_float('dropout', 0.3, 0.55)
    
    model = InfluencerModelMultiBranch(
        tweet_dim=TWEET_DIM,
        desc_dim=DESC_DIM,
        meta_dim=FEAT_DIM,
        hidden_tweet=hidden_tweet,
        hidden_desc=hidden_desc,
        hidden_feat=hidden_feat,
        fusion_dim=fusion_dim,
        n_classes=2,
        dropout=dropout
    )
    
    return model


def objective(trial):
    """Optuna objective function with AGGRESSIVE anti-overfitting.
    
    Returns a penalized score that discourages overfitting:
    - Monitors train-val F1 gap
    - Prunes trials with gap > 4%
    - Penalizes score based on gap magnitude
    """
    
    # Optimizer hyperparameters - CONSTRAINED for regularization
    lr = trial.suggest_float('lr', 5e-5, 3e-3, log=True)
    weight_decay = trial.suggest_float('weight_decay', 0.5, 10, log=True)
    batch_size = trial.suggest_categorical('batch_size', [256, 512])
    
    # Create model
    model = create_model_from_trial(trial).to(device)
    
    # Create data loaders with suggested batch size
    train_dataset_opt = MultiBranchDataset(
        X_train_text_split, X_train_desc_split, X_train_feat_split, y_train
    )
    val_dataset_opt = MultiBranchDataset(
        X_val_text_split, X_val_desc_split, X_val_feat_split, y_val
    )
    
    train_loader_opt = DataLoader(
        train_dataset_opt, batch_size=batch_size, shuffle=True,
        num_workers=4, pin_memory=True
    )
    val_loader_opt = DataLoader(
        val_dataset_opt, batch_size=batch_size, shuffle=False,
        num_workers=4, pin_memory=True
    )
    
    # Criterion with class weights
    class_counts = np.bincount(y_train)
    class_weights = torch.tensor([1.0 / c for c in class_counts], dtype=torch.float32).to(device)
    class_weights = class_weights / class_weights.sum() * 2
    criterion = nn.CrossEntropyLoss(weight=class_weights)
    
    # Optimizer and scheduler
    optimizer = AdamW(model.parameters(), lr=lr, weight_decay=weight_decay)
    total_steps = len(train_loader_opt) * OPTUNA_EPOCHS
    scheduler = OneCycleLR(
        optimizer, max_lr=lr, total_steps=total_steps,
        pct_start=0.1, anneal_strategy='cos'
    )
    
    # Training loop with early stopping and gap monitoring
    best_val_f1 = 0
    best_gap = 0
    patience_counter = 0
    
    for epoch in range(OPTUNA_EPOCHS):
        # Train
        model.train()
        for batch in train_loader_opt:
            tweet, desc, features, labels = [b.to(device) for b in batch]
            optimizer.zero_grad()
            logits = model(tweet, desc, features)
            loss = criterion(logits, labels)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()
            scheduler.step()
        
        # Evaluate BOTH train and val for gap monitoring
        train_f1 = evaluate_f1(model, train_loader_opt, device)
        val_f1 = evaluate_f1(model, val_loader_opt, device)
        gap = train_f1 - val_f1
        
        # AGGRESSIVE: Prune immediately if gap too large
        if gap > MAX_TRAIN_VAL_GAP:
            print(f"  Trial {trial.number} pruned at epoch {epoch}: gap={gap:.3f} > {MAX_TRAIN_VAL_GAP}")
            raise optuna.TrialPruned()
        
        # Report intermediate value for pruning (penalized score)
        # ULTRA-STRICT: Heavy penalty for any gap above target
        penalized_score = val_f1 - GAP_PENALTY_WEIGHT * max(0, gap - GAP_TARGET)
        trial.report(penalized_score, epoch)
        
        # Handle standard pruning
        if trial.should_prune():
            raise optuna.TrialPruned()
        
        # Track best (using penalized score)
        best_penalized = best_val_f1 - GAP_PENALTY_WEIGHT * max(0, best_gap - GAP_TARGET)
        if penalized_score > best_penalized:
            best_val_f1 = val_f1
            best_gap = gap
            patience_counter = 0
        else:
            patience_counter += 1
            if patience_counter >= 3:
                break
    
    # Cleanup
    del model, optimizer, train_loader_opt, val_loader_opt
    gc.collect()
    torch.cuda.empty_cache()
    
    # ULTRA-STRICT: Return heavily penalized score
    final_score = best_val_f1 - GAP_PENALTY_WEIGHT * max(0, best_gap - GAP_TARGET)
    
    print(f"  Trial {trial.number}: val_f1={best_val_f1:.4f}, gap={best_gap:.4f}, score={final_score:.4f}")
    
    return final_score

print("ULTRA-STRICT anti-overfitting objective function defined")
print(f"   HARD REJECT if gap > {MAX_TRAIN_VAL_GAP*100:.1f}%")
print(f"   Target gap: {GAP_TARGET*100:.1f}%")
print(f"   Penalty weight: {GAP_PENALTY_WEIGHT}")

In [ ]:
# =============================================================================
# RUN OPTUNA OPTIMIZATION
# =============================================================================

if RUN_OPTUNA:
    print("Starting Optuna Hyperparameter Optimization...")
    print(f"   Trials: {N_TRIALS}, Epochs per trial: {OPTUNA_EPOCHS}")
    print(f"   Timeout: {TIMEOUT/3600:.1f} hours")
    print("=" * 60)
    
    # Create study with pruning
    study = optuna.create_study(
        study_name=STUDY_NAME,
        direction='maximize',  # Maximize F1 score
        pruner=optuna.pruners.MedianPruner(n_startup_trials=5, n_warmup_steps=3),
        sampler=optuna.samplers.TPESampler(seed=SEED)
    )
    
    # Run optimization
    study.optimize(
        objective,
        n_trials=N_TRIALS,
        timeout=TIMEOUT,
        show_progress_bar=True,
        gc_after_trial=True
    )
    
    # Print results
    print("\n" + "=" * 60)
    print("OPTIMIZATION COMPLETE")
    print("=" * 60)
    
    print(f"\nBest trial:")
    print(f"   F1 Score: {study.best_value:.4f}")
    print(f"   Trial number: {study.best_trial.number}")
    
    print(f"\nBest hyperparameters:")
    for key, value in study.best_params.items():
        print(f"   {key}: {value}")
    
    # Save best params
    best_params = study.best_params
    joblib.dump(best_params, 'models/optuna_best_params.joblib')
    print(f"\nBest params saved to models/optuna_best_params.joblib")
    
    # Statistics
    pruned_trials = len([t for t in study.trials if t.state == TrialState.PRUNED])
    complete_trials = len([t for t in study.trials if t.state == TrialState.COMPLETE])
    print(f"\nTrial statistics:")
    print(f"   Completed: {complete_trials}")
    print(f"   Pruned: {pruned_trials}")
    
else:
    print("Skipping Optuna optimization (RUN_OPTUNA=False)")
    print("   Set RUN_OPTUNA=True to run optimization")
    
    # Try to load saved params
    if os.path.exists('models/optuna_best_params.joblib'):
        best_params = joblib.load('models/optuna_best_params.joblib')
        print(f"\nLoaded saved best params:")
        for key, value in best_params.items():
            print(f"   {key}: {value}")
    else:
        best_params = None
        print("   No saved params found - using defaults")

In [ ]:
# =============================================================================
# OPTUNA VISUALIZATION
# =============================================================================

if RUN_OPTUNA and 'study' in dir():
    print("Optuna Optimization Visualizations")
    print("=" * 60)
    
    try:
        # 1. Optimization history
        fig1 = optuna.visualization.plot_optimization_history(study)
        fig1.update_layout(title="Optimization History - F1 Score over Trials")
        fig1.show()
        
        # 2. Parameter importance
        fig2 = optuna.visualization.plot_param_importances(study)
        fig2.update_layout(title="Hyperparameter Importance")
        fig2.show()
        
        # 3. Parallel coordinate plot
        fig3 = optuna.visualization.plot_parallel_coordinate(study)
        fig3.update_layout(title="Parallel Coordinate Plot")
        fig3.show()
        
        # 4. Slice plot for key parameters
        fig4 = optuna.visualization.plot_slice(study, params=['lr', 'dropout', 'hidden_tweet'])
        fig4.update_layout(title="Parameter Slice Plot")
        fig4.show()
        
    except Exception as e:
        print(f"Visualization error: {e}")
        print("   Install plotly for interactive visualizations: pip install plotly")
        
        # Fallback: matplotlib visualization
        print("\nTrial Results (matplotlib fallback):")
        trials_df = study.trials_dataframe()
        
        fig, axes = plt.subplots(1, 2, figsize=(14, 5))
        
        # F1 over trials
        axes[0].plot(trials_df['number'], trials_df['value'], 'b-o', alpha=0.7)
        axes[0].axhline(y=study.best_value, color='r', linestyle='--', label=f'Best: {study.best_value:.4f}')
        axes[0].set_xlabel('Trial')
        axes[0].set_ylabel('F1 Score')
        axes[0].set_title('Optimization History')
        axes[0].legend()
        axes[0].grid(True, alpha=0.3)
        
        # Best params bar chart
        params = list(study.best_params.keys())
        values = [str(v)[:10] for v in study.best_params.values()]
        axes[1].barh(params, range(len(params)))
        for i, (p, v) in enumerate(zip(params, values)):
            axes[1].text(i + 0.1, i, f"{v}", va='center')
        axes[1].set_xlabel('Value')
        axes[1].set_title('Best Hyperparameters')
        
        plt.tight_layout()
        plt.show()

else:
    print("Skipping visualization (Optuna not run)")
    print("   Set RUN_OPTUNA=True and run the optimization first")

In [ ]:
# =============================================================================
# USE BEST HYPERPARAMETERS (from Optuna or defaults)
# =============================================================================

# Default hyperparameters
DEFAULT_PARAMS = {
    'lr': 1e-3,
    'weight_decay': 4.0,
    'dropout': 0.3,
    'hidden_tweet': 512,
    'hidden_desc': 512,
    'hidden_feat': 128,
    'fusion_dim': 256,
    'batch_size': 512
}

# Use Optuna params if available, otherwise defaults
if best_params is not None:
    print("Using Optuna-optimized hyperparameters:")
    FINAL_PARAMS = {**DEFAULT_PARAMS, **best_params}
else:
    print("Using default hyperparameters:")
    FINAL_PARAMS = DEFAULT_PARAMS

for key, value in FINAL_PARAMS.items():
    print(f"   {key}: {value}")

# Create final model with best/default params using factory function
model = create_model(
    MODEL_ARCHITECTURE,
    tweet_dim=TWEET_DIM,
    desc_dim=DESC_DIM,
    meta_dim=FEAT_DIM,
    hidden_tweet=FINAL_PARAMS.get('hidden_tweet', 512),
    hidden_desc=FINAL_PARAMS.get('hidden_desc', 512),
    hidden_feat=FINAL_PARAMS.get('hidden_feat', 128),
    fusion_dim=FINAL_PARAMS.get('fusion_dim', 256),
    n_classes=2,
    dropout=FINAL_PARAMS['dropout']
).to(device)

print(f"\nFinal Model Architecture ({MODEL_ARCHITECTURE.upper()}):")
print(f"   Total parameters: {sum(p.numel() for p in model.parameters()):,}")

# Update batch size if needed
if FINAL_PARAMS['batch_size'] != batch_size:
    batch_size = FINAL_PARAMS['batch_size']
    train_loader, val_loader, train_dataset, val_dataset = create_datasets_and_loaders(
        MODEL_ARCHITECTURE, batch_size
    )
    print(f"   Updated batch size: {batch_size}")

# Update training config
LR = FINAL_PARAMS['lr']
WEIGHT_DECAY = FINAL_PARAMS['weight_decay']
EPOCHS = 30  # Full training with more epochs

# Criterion
class_counts = np.bincount(y_train)
class_weights = torch.tensor([1.0 / c for c in class_counts], dtype=torch.float32).to(device)
class_weights = class_weights / class_weights.sum() * 2
criterion = nn.CrossEntropyLoss(weight=class_weights)

# Optimizer and scheduler
optimizer = AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
total_steps = len(train_loader) * EPOCHS
scheduler = OneCycleLR(
    optimizer, max_lr=LR, total_steps=total_steps,
    pct_start=0.1, anneal_strategy='cos'
)

print(f"\nReady for final training with {'optimized' if best_params else 'default'} hyperparameters!")

In [ ]:
# =============================================================================
# RUN TRAINING
# =============================================================================

history = {
    'train_loss': [], 'val_loss': [],
    'train_acc': [], 'val_acc': [],
    'train_f1': [], 'val_f1': []
}

best_val_acc = 0
best_val_f1 = 0
patience_counter = 0
best_model_state = None

print("\n" + "="*70)
print("TRAINING STARTED")
print("="*70)

for epoch in range(1, EPOCHS + 1):
    # Train
    train_loss, train_acc, train_f1 = train_epoch(
        model, train_loader, optimizer, criterion, scheduler, device, MODEL_ARCHITECTURE
    )
    
    # Evaluate
    val_loss, val_acc, val_f1, val_probs = evaluate(
        model, val_loader, criterion, device, MODEL_ARCHITECTURE
    )
    
    # Store history
    history['train_loss'].append(train_loss)
    history['val_loss'].append(val_loss)
    history['train_acc'].append(train_acc)
    history['val_acc'].append(val_acc)
    history['train_f1'].append(train_f1)
    history['val_f1'].append(val_f1)
    
    # Print progress
    print(f"Epoch {epoch:2d}/{EPOCHS} | "
          f"Train Loss: {train_loss:.4f} Acc: {train_acc:.4f} F1: {train_f1:.4f} | "
          f"Val Loss: {val_loss:.4f} Acc: {val_acc:.4f} F1: {val_f1:.4f}")
    
    # Save best model
    if val_f1 > best_val_f1:
        best_val_f1 = val_f1
        best_val_acc = val_acc
        best_model_state = model.state_dict().copy()
        patience_counter = 0
        print(f"   [NEW BEST] Val F1: {val_f1:.4f}")
    else:
        patience_counter += 1
        if patience_counter >= PATIENCE:
            print(f"\nEarly stopping at epoch {epoch}")
            break

print("\n" + "="*70)
print(f"Best model - Val Accuracy: {best_val_acc:.4f}, Val F1: {best_val_f1:.4f}")
print("="*70)

# Restore best model
if best_model_state is not None:
    model.load_state_dict(best_model_state)

In [ ]:
# Plot training history
plot_training_history(history)

## 5. Gradient Boosting Models (Ensemble Diversity)

In [ ]:
# =============================================================================
# XGBOOST & LIGHTGBM (GPU + CPU optimized) - WITH LOADING
# =============================================================================

# Hardware configuration
N_THREADS = 20  # i7-14700K: 20 physical cores

# ============= SCALER (uses combined data for boosting) =============
if TRAIN_XGBOOST or TRAIN_LIGHTGBM:
    if os.path.exists(SCALER_PATH) and not TRAIN_XGBOOST and not TRAIN_LIGHTGBM:
        scaler = joblib.load(SCALER_PATH)
        X_train_scaled = scaler.transform(X_train_comb)
        X_val_scaled = scaler.transform(X_val_comb)
        print("Scaler loaded from checkpoint")
    else:
        scaler = StandardScaler()
        X_train_scaled = scaler.fit_transform(X_train_comb)
        X_val_scaled = scaler.transform(X_val_comb)
        print("New scaler created and fitted")
else:
    scaler = joblib.load(SCALER_PATH)
    X_train_scaled = scaler.transform(X_train_comb)
    X_val_scaled = scaler.transform(X_val_comb)
    print("Scaler loaded from checkpoint")

# ============= XGBOOST =============
if TRAIN_XGBOOST:
    print("\nTraining XGBoost (GPU: RTX 4000 Ada)...")
    xgb_model = XGBClassifier(
        n_estimators=1000,
        max_depth=8,
        learning_rate=0.05,
        subsample=0.8,
        colsample_bytree=0.8,
        random_state=SEED,
        eval_metric='logloss',
        early_stopping_rounds=50,
        tree_method='hist',      # GPU optimized
        device='cuda',           # RTX 4000
        max_bin=256,
    )
    
    xgb_model.fit(
        X_train_scaled, y_train,
        eval_set=[(X_val_scaled, y_val)],
        verbose=True
    )
    
    xgb_preds = xgb_model.predict(X_val_scaled)
    xgb_probs = xgb_model.predict_proba(X_val_scaled)
    print(f"   XGBoost Val Accuracy: {accuracy_score(y_val, xgb_preds):.4f}")
    print(f"   XGBoost Val F1: {f1_score(y_val, xgb_preds, average='macro'):.4f}")
else:
    print("\nLoading XGBoost from checkpoint...")
    xgb_model = joblib.load(XGBOOST_PATH)
    xgb_preds = xgb_model.predict(X_val_scaled)
    xgb_probs = xgb_model.predict_proba(X_val_scaled)
    print(f"   [OK] Loaded - Val Accuracy: {accuracy_score(y_val, xgb_preds):.4f}")
    print(f"                Val F1: {f1_score(y_val, xgb_preds, average='macro'):.4f}")

# ============= LIGHTGBM =============
if TRAIN_LIGHTGBM:
    print(f"\nTraining LightGBM (CPU: {N_THREADS} threads)...")
    lgbm_model = LGBMClassifier(
        n_estimators=500,
        max_depth=8,
        num_leaves=63,
        learning_rate=0.05,
        subsample=0.8,
        colsample_bytree=0.8,
        random_state=SEED,
        n_jobs=N_THREADS,        # All cores
        verbose=-1
    )
    
    lgbm_model.fit(
        X_train_scaled, y_train,
        eval_set=[(X_val_scaled, y_val)]
    )
    
    lgbm_preds = lgbm_model.predict(X_val_scaled)
    lgbm_probs = lgbm_model.predict_proba(X_val_scaled)
    print(f"   LightGBM Val Accuracy: {accuracy_score(y_val, lgbm_preds):.4f}")
    print(f"   LightGBM Val F1: {f1_score(y_val, lgbm_preds, average='macro'):.4f}")
else:
    print("\nLoading LightGBM from checkpoint...")
    lgbm_model = joblib.load(LIGHTGBM_PATH)
    lgbm_preds = lgbm_model.predict(X_val_scaled)
    lgbm_probs = lgbm_model.predict_proba(X_val_scaled)
    print(f"   [OK] Loaded - Val Accuracy: {accuracy_score(y_val, lgbm_preds):.4f}")
    print(f"                Val F1: {f1_score(y_val, lgbm_preds, average='macro'):.4f}")

## 6. Ensemble (Weighted Voting)

In [ ]:
# =============================================================================
# ENSEMBLE - WEIGHTED VOTING
# =============================================================================

# Get Neural Network predictions
_, nn_val_acc, nn_val_f1, nn_val_probs = evaluate(model, val_loader, criterion, device, MODEL_ARCHITECTURE)

# Ensemble weights (based on validation F1)
weights = {
    'nn': nn_val_f1,
    'xgb': f1_score(y_val, xgb_preds, average='macro'),
    'lgbm': f1_score(y_val, lgbm_preds, average='macro')
}

# Normalize weights
total_weight = sum(weights.values())
weights = {k: v / total_weight for k, v in weights.items()}

print("\nEnsemble Weights:")
for name, w in weights.items():
    print(f"   {name}: {w:.4f}")

# Weighted average of probabilities
ensemble_probs = (
    weights['nn'] * nn_val_probs +
    weights['xgb'] * xgb_probs +
    weights['lgbm'] * lgbm_probs
)

ensemble_preds = np.argmax(ensemble_probs, axis=1)
ensemble_acc = accuracy_score(y_val, ensemble_preds)
ensemble_f1 = f1_score(y_val, ensemble_preds, average='macro')

print(f"\nEnsemble Results:")
print(f"   Accuracy: {ensemble_acc:.4f}")
print(f"   F1 Score: {ensemble_f1:.4f}")

# Compare all models
print("\n" + "="*50)
print("MODEL COMPARISON")
print("="*50)
print(f"{'Model':<15} {'Accuracy':>10} {'F1 Score':>10}")
print("-"*35)
print(f"{'Neural Net':<15} {nn_val_acc:>10.4f} {nn_val_f1:>10.4f}")
print(f"{'XGBoost':<15} {accuracy_score(y_val, xgb_preds):>10.4f} {f1_score(y_val, xgb_preds, average='macro'):>10.4f}")
print(f"{'LightGBM':<15} {accuracy_score(y_val, lgbm_preds):>10.4f} {f1_score(y_val, lgbm_preds, average='macro'):>10.4f}")
print(f"{'Ensemble':<15} {ensemble_acc:>10.4f} {ensemble_f1:>10.4f}")
print("="*50)

## 7. Generate Kaggle Submission

In [ ]:
# =============================================================================
# GENERATE KAGGLE SUBMISSION (Architecture-aware)
# =============================================================================

# Scale Kaggle data for boosting models
X_kaggle_scaled = scaler.transform(X_kaggle_combined)

# Create Kaggle dataset for NN based on architecture
if MODEL_ARCHITECTURE == "simple":
    X_kaggle_all = np.hstack([X_kaggle_text, X_kaggle_desc, X_kaggle_feat])
    kaggle_dataset = SimpleDataset(X_kaggle_all, labels=None)
else:  # multibranch
    kaggle_dataset = MultiBranchDataset(X_kaggle_text, X_kaggle_desc, X_kaggle_feat, labels=None)

kaggle_loader = DataLoader(kaggle_dataset, batch_size=batch_size, shuffle=False,
                           num_workers=2, pin_memory=True)

# Get predictions from all models
print(f"\nGenerating predictions ({MODEL_ARCHITECTURE.upper()})...")

# Neural Network predictions
model.eval()
nn_kaggle_probs = []
with torch.no_grad():
    for batch in kaggle_loader:
        if MODEL_ARCHITECTURE == "simple":
            features = batch.to(device)
            logits = model(features)
        else:  # multibranch
            tweet, desc, features = [b.to(device) for b in batch]
            logits = model(tweet, desc, features)
        probs = F.softmax(logits, dim=1)
        nn_kaggle_probs.extend(probs.cpu().numpy())
nn_kaggle_probs = np.array(nn_kaggle_probs)

# XGBoost & LightGBM (use combined features)
xgb_kaggle_probs = xgb_model.predict_proba(X_kaggle_scaled)
lgbm_kaggle_probs = lgbm_model.predict_proba(X_kaggle_scaled)

# Ensemble
ensemble_kaggle_probs = (
    weights['nn'] * nn_kaggle_probs +
    weights['xgb'] * xgb_kaggle_probs +
    weights['lgbm'] * lgbm_kaggle_probs
)
ensemble_kaggle_preds = np.argmax(ensemble_kaggle_probs, axis=1)

print(f"   Total predictions: {len(ensemble_kaggle_preds)}")
print(f"   Class distribution: {np.bincount(ensemble_kaggle_preds)}")

In [ ]:
# =============================================================================
# SAVE SUBMISSION
# =============================================================================

# Load test data for IDs
kaggle_df = pd.read_json('data/kaggle_test.jsonl', lines=True)
kaggle_df = pd.json_normalize(kaggle_df.to_dict(orient='records'))

# Get challenge IDs
ids = kaggle_df['challenge_id'].astype(int).values

# Create submission DataFrame
submission = pd.DataFrame({
    'ID': ids,
    'Prediction': ensemble_kaggle_preds
})

# Save
os.makedirs('submission', exist_ok=True)
submission_path = 'submission/submission_ensemble_model.csv'
submission.to_csv(submission_path, index=False)

print(f"\nSubmission saved to: {submission_path}")
print(f"   Shape: {submission.shape}")
print(f"   Class distribution: {submission['Prediction'].value_counts().to_dict()}")
print("\nSample predictions:")
print(submission.head(10))

## 8. Neural Network Solo Submission

Generates a submission with only the optimized NN, to compare easily to the ensemble method.

In [ ]:
# =============================================================================
# NEURAL NETWORK SOLO SUBMISSION (Architecture-aware)
# =============================================================================

print(f"Generating {MODEL_ARCHITECTURE.upper()} Neural Network predictions...")
print(f"   Using best model: Val F1 = {best_val_f1:.4f}, Val Acc = {best_val_acc:.4f}")

# Make sure we're using the best model (already loaded from training)
model.eval()

# Create Kaggle dataset based on architecture
if MODEL_ARCHITECTURE == "simple":
    X_kaggle_all = np.hstack([X_kaggle_text, X_kaggle_desc, X_kaggle_feat])
    kaggle_dataset_nn = SimpleDataset(X_kaggle_all, labels=None)
else:  # multibranch
    kaggle_dataset_nn = MultiBranchDataset(X_kaggle_text, X_kaggle_desc, X_kaggle_feat, labels=None)

kaggle_loader_nn = DataLoader(
    kaggle_dataset_nn, 
    batch_size=batch_size, 
    shuffle=False,
    num_workers=4, 
    pin_memory=True
)

# Generate predictions
nn_solo_probs = []
with torch.no_grad():
    for batch in kaggle_loader_nn:
        if MODEL_ARCHITECTURE == "simple":
            features = batch.to(device)
            logits = model(features)
        else:  # multibranch
            tweet, desc, features = [b.to(device) for b in batch]
            logits = model(tweet, desc, features)
        probs = F.softmax(logits, dim=1)
        nn_solo_probs.extend(probs.cpu().numpy())

nn_solo_probs = np.array(nn_solo_probs)
nn_solo_preds = np.argmax(nn_solo_probs, axis=1)

print(f"   Predictions generated: {len(nn_solo_preds)}")
print(f"   Class distribution: {np.bincount(nn_solo_preds)}")
print(f"      - Observers (0): {np.sum(nn_solo_preds == 0)} ({np.sum(nn_solo_preds == 0)/len(nn_solo_preds)*100:.1f}%)")
print(f"      - Influencers (1): {np.sum(nn_solo_preds == 1)} ({np.sum(nn_solo_preds == 1)/len(nn_solo_preds)*100:.1f}%)")

# =============================================================================
# SAVE NEURAL NETWORK SUBMISSION
# =============================================================================

# Load test data for IDs
kaggle_test_df = pd.read_json('data/kaggle_test.jsonl', lines=True)
kaggle_test_df = pd.json_normalize(kaggle_test_df.to_dict(orient='records'))

# Get challenge IDs
ids = kaggle_test_df['challenge_id'].astype(int).values

# Create submission DataFrame
nn_submission = pd.DataFrame({
    'ID': ids,
    'Prediction': nn_solo_preds
})

# Save to file
os.makedirs('submission', exist_ok=True)
nn_submission_path = 'submission/submission_neural_network_solo.csv'
nn_submission.to_csv(nn_submission_path, index=False)

print(f"\nNeural Network submission saved!")
print(f"   Path: {nn_submission_path}")
print(f"   Shape: {nn_submission.shape}")
print(f"\nFirst 10 predictions:")
print(nn_submission.head(10))

print(f"\nModel Performance Summary:")
print(f"   Validation F1: {best_val_f1:.4f} [BEST]")
print(f"   Validation Accuracy: {best_val_acc:.4f}")
print(f"   (vs Ensemble F1: {ensemble_f1:.4f})")
print(f"\nThis neural network outperformed the ensemble by {(best_val_f1 - ensemble_f1)*100:.2f}% F1!")

## 9. Summary and Next Steps

### This notebook combines:
1. **Lab4 Techniques**: Dropout, Weight Decay, Gradient Clipping
2. **Lab5 Techniques**: Transfer Learning via CamemBERT embeddings
3. **Lab6 Techniques**: Feature Engineering, interaction features
4. **Lab8 Techniques**: AdamW optimizer, Learning Rate Scheduling

### Possible improvements:
- Fine-tuning CamemBERT end-to-end (if GPU available)
- Pseudo-labeling on confident predictions
- Cross-validation for more robust ensemble weights
- Test-Time Augmentation (TTA)

In [ ]:
# =============================================================================
# SAVE MODEL CHECKPOINT (Multi-Branch)
# =============================================================================

os.makedirs('models', exist_ok=True)

# Save Multi-Branch Neural Network
checkpoint = {
    'model_state_dict': model.state_dict(),
    'tweet_dim': TWEET_DIM,
    'desc_dim': DESC_DIM,
    'meta_dim': FEAT_DIM,
    'best_val_acc': best_val_acc,
    'best_val_f1': best_val_f1,
    'weights': weights
}
torch.save(checkpoint, 'models/influencer_model_multibranch.pt')

# Save XGBoost & LightGBM
import joblib
joblib.dump(xgb_model, 'models/xgb_model.joblib')
joblib.dump(lgbm_model, 'models/lgbm_model.joblib')
joblib.dump(scaler, 'models/scaler.joblib')

print("All models saved (Multi-Branch)!")